# Dataset preparation
The high level task: take the data that was already processed with the Lapa LLM and based on the result data create low-level model to classify the paragraphs depending on the parts. 

In [18]:
import pandas as pd
import numpy as np
from tempfile import TemporaryFile, TemporaryDirectory
from pathlib import Path
from tqdm.notebook import tqdm

In [19]:
BUCKET_DATA = "court_data_2024_structured"

In [20]:
from google.cloud import bigquery
from google.cloud import storage

In [21]:
bq_client = bigquery.Client()
storage_client = storage.Client()

/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/p

Extract the data that already has classification

In [22]:
CLASSIFICATION_DOCS_QUERY = """
SELECT
    justice_kind,
    doc_id,
FROM `lab-test-project-1-305710.court_data_2024.document_data`
WHERE is_classified = true
"""

In [23]:
classification_docs_df = bq_client.query(CLASSIFICATION_DOCS_QUERY).to_dataframe()

/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [24]:
classification_docs_df

,justice_kind,doc_id
0,2,116096838
1,2,116096844
2,2,116098806
3,2,116070816
4,2,116085232
...,...,...
8506,2,116097181
8507,2,116097397
8508,2,116087469
8509,2,116090636


In [25]:
def download_parquet_data(justice_kind, doc_id: str, download_path: str):
    storage_path = f"document_text_parsing/info_version_9/{justice_kind}/{doc_id}/classification.parquet"
    blob = storage_client.bucket(BUCKET_DATA).blob(storage_path)
    blob.download_to_filename(download_path)

In [26]:
pd.read_parquet("temp_data/temp.parquet")

,document_id,paragraph_index,paragraph_order,numbered_text,text,section
0,116096838,1,1,[paragraph_id=1] ХАРКІВСЬКИЙ АПЕЛЯЦІЙНИЙ СУД,ХАРКІВСЬКИЙ АПЕЛЯЦІЙНИЙ СУД,introductory
1,116096838,2,2,[paragraph_id=2] _____________________________...,______________________________________________...,introductory
2,116096838,3,3,[paragraph_id=3] Справа № 818/1367т/23,Справа № 818/1367т/23,introductory
3,116096838,4,4,[paragraph_id=4] Провадження № 11кс/818/1321/23,Провадження № 11кс/818/1321/23,introductory
4,116096838,5,5,[paragraph_id=5] УХВАЛА,УХВАЛА,introductory
5,116096838,6,6,[paragraph_id=6] Іменем України,Іменем України,introductory
6,116096838,7,7,[paragraph_id=7] про дозвіл на проведення негл...,про дозвіл на проведення негласної слідчої (ро...,introductory
7,116096838,8,8,[paragraph_id=8] м. Харків ...,м. Харків ...,introductory
8,116096838,9,9,[paragraph_id=9] Слідчий суддя Харківського ап...,Слідчий суддя Харківського апеляційного суду О...,introductory
9,116096838,10,10,[paragraph_id=10] В С Т А Н О В И В:,В С Т А Н О В И В:,reasoning


In [27]:
def compose_dataset(doc_id):
    with TemporaryDirectory(prefix="legal_p_") as temp_dir:
        temp_dir = Path(temp_dir)
        doc_temp_path = f"doc_{doc_id}.parquet"
        local_path = temp_dir / doc_temp_path
        print(local_path)

        download_parquet_data(2, doc_id, str(local_path))
        return pd.read_parquet(doc_temp_path)

In [28]:
with TemporaryDirectory(prefix="legal_p_") as temp_dir:
    paragraphs = []
    for row in tqdm(classification_docs_df.itertuples(index=False), total=classification_docs_df.shape[0]):
        doc_id = str(row[1])
        temp_dir = Path(temp_dir)
        doc_temp_path = f"doc_{str(doc_id)}.parquet"
        local_path = temp_dir / doc_temp_path
        print(doc_id, " ->  ", local_path)

        download_parquet_data(int(row[0]), doc_id, str(local_path))
        paragraphs.append(pd.read_parquet(local_path))

  0%|          | 0/8511 [00:00<?, ?it/s]

116096838  ->   /tmp/legal_p_8r9m1g1j/doc_116096838.parquet
116096844  ->   /tmp/legal_p_8r9m1g1j/doc_116096844.parquet
116098806  ->   /tmp/legal_p_8r9m1g1j/doc_116098806.parquet
116070816  ->   /tmp/legal_p_8r9m1g1j/doc_116070816.parquet
116085232  ->   /tmp/legal_p_8r9m1g1j/doc_116085232.parquet
116076003  ->   /tmp/legal_p_8r9m1g1j/doc_116076003.parquet
116107429  ->   /tmp/legal_p_8r9m1g1j/doc_116107429.parquet
116070703  ->   /tmp/legal_p_8r9m1g1j/doc_116070703.parquet
116084207  ->   /tmp/legal_p_8r9m1g1j/doc_116084207.parquet
116091943  ->   /tmp/legal_p_8r9m1g1j/doc_116091943.parquet
116091717  ->   /tmp/legal_p_8r9m1g1j/doc_116091717.parquet
116069331  ->   /tmp/legal_p_8r9m1g1j/doc_116069331.parquet
116083274  ->   /tmp/legal_p_8r9m1g1j/doc_116083274.parquet
116098506  ->   /tmp/legal_p_8r9m1g1j/doc_116098506.parquet
116103559  ->   /tmp/legal_p_8r9m1g1j/doc_116103559.parquet
116103586  ->   /tmp/legal_p_8r9m1g1j/doc_116103586.parquet
116072132  ->   /tmp/legal_p_8r9m1g1j/do

In [29]:
p_df = pd.concat(paragraphs, sort=False)

In [30]:
p_df

,document_id,paragraph_index,paragraph_order,numbered_text,text,section
0,116096838,1,1,[paragraph_id=1] ХАРКІВСЬКИЙ АПЕЛЯЦІЙНИЙ СУД,ХАРКІВСЬКИЙ АПЕЛЯЦІЙНИЙ СУД,introductory
1,116096838,2,2,[paragraph_id=2] _____________________________...,______________________________________________...,introductory
2,116096838,3,3,[paragraph_id=3] Справа № 818/1367т/23,Справа № 818/1367т/23,introductory
3,116096838,4,4,[paragraph_id=4] Провадження № 11кс/818/1321/23,Провадження № 11кс/818/1321/23,introductory
4,116096838,5,5,[paragraph_id=5] УХВАЛА,УХВАЛА,introductory
...,...,...,...,...,...,...
43,116097621,44,44,[paragraph_id=44] П О С Т А Н О В И Л А:,П О С Т А Н О В И Л А:,operative
44,116097621,45,45,[paragraph_id=45] Заяву засудженого ОСОБА_6 пр...,Заяву засудженого ОСОБА_6 про перегляд за ново...,operative
45,116097621,46,46,[paragraph_id=46] Ухвала набирає чинності з мо...,Ухвала набирає чинності з моменту її проголоше...,operative
46,116097621,47,47,[paragraph_id=47] Судді:,Судді:,operative


In [33]:
p_df.shape, p_df.columns

((289367, 6),
 Index(['document_id', 'paragraph_index', 'paragraph_order', 'numbered_text',
        'text', 'section'],
       dtype='str'))

In [32]:
p_df.to_parquet("legal_paragraphs.parquet")

In [34]:
%reset